# IMPORT LIBRARY

In [1]:
# ==========================================
# CELL 1: IMPORT & INISIALISASI CUDA
# ==========================================
import pandas as pd
import numpy as np
import torch
import re
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score
import nlpaug.augmenter.word as naw
from tqdm import tqdm # Untuk progress bar

import dagshub
import mlflow
import mlflow.sklearn

# Cek GPU (CUDA)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Menggunakan perangkat: {device}")
if device.type != 'cuda':
    print("⚠️ PERINGATAN: Anda belum mengaktifkan GPU. Kecepatan IndoBERT akan sangat lambat!")
else:
    print(f"✅ GPU Aktif: {torch.cuda.get_device_name(0)}")

d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Menggunakan perangkat: cuda
✅ GPU Aktif: NVIDIA GeForce RTX 3060 Laptop GPU


# IMPORT DATASET

In [2]:
# ==========================================
# CELL 2: LOAD DATA BARU & CLEANING
# ==========================================
file_path = "../../data/dataset_keluhan_ntg.csv" # Sesuaikan path

# 1. Load Data
df = pd.read_csv(file_path, sep='|', on_bad_lines='skip')

# 2. Definisikan X dan Y (Gunakan jenis_kerusakan)
kolom_target = ['kategori_aset', 'severity', 'root_cause', 'jenis_kerusakan']
df = df.dropna(subset=kolom_target + ['teks_keluhan_awam', 'teks_laporan_teknisi'])

# Gabungkan teks keluhan awam dan laporan teknisi
df['input_teks'] = df['teks_keluhan_awam'].astype(str) + " " + df['teks_laporan_teknisi'].astype(str)

Y = df[kolom_target]
X_raw = df['input_teks']

# 3. Basic Cleaning (Hanya hapus simbol aneh, biarkan kata utuh)
def basic_clean(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text) # Sisakan huruf dan angka
    text = re.sub(r'\s+', ' ', text).strip()
    return text

X_clean = X_raw.apply(basic_clean)
print(f"✅ Cell 2 Selesai: {len(X_clean)} baris teks dibersihkan tanpa menghilangkan konteks asli.")

✅ Cell 2 Selesai: 415 baris teks dibersihkan tanpa menghilangkan konteks asli.


# DATA AUGMENTATION

In [3]:
# ==========================================
# CELL 3: CONTEXTUAL DATA AUGMENTATION (INDOBERT)
# ==========================================
print("⏳ Memulai Augmentasi Data (Menggandakan variasi kalimat)...")

# Inisialisasi Augmenter berbasis IndoBERT (Berjalan di GPU)
aug = naw.ContextualWordEmbsAug(
    model_path='indobenchmark/indobert-base-p1', 
    action="substitute", 
    device='cuda' if torch.cuda.is_available() else 'cpu',
    aug_p=0.2 # Ubah 20% kata di setiap kalimat
)

X_aug_list = []
Y_aug_list = []

# Proses Augmentasi
for i, teks in enumerate(tqdm(X_clean, desc="Augmentasi")):
    # Simpan teks asli
    X_aug_list.append(teks)
    Y_aug_list.append(Y.iloc[i].values)
    
    # Buat 1 variasi teks baru per kalimat asli
    teks_baru = aug.augment(teks)
    # Jika nlpaug mengembalikan list, ambil elemen pertamanya
    if isinstance(teks_baru, list):
        teks_baru = teks_baru[0]
        
    X_aug_list.append(teks_baru)
    Y_aug_list.append(Y.iloc[i].values)

X_final = pd.Series(X_aug_list)
Y_final = pd.DataFrame(Y_aug_list, columns=kolom_target)

print(f"✅ Augmentasi Selesai! Data melonjak dari {len(X_clean)} menjadi {len(X_final)} baris.")

⏳ Memulai Augmentasi Data (Menggandakan variasi kalimat)...


d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\transformers\modeling_utils.py:446: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer b

✅ Augmentasi Selesai! Data melonjak dari 415 menjadi 830 baris.


In [4]:
# ==========================================
# CELL 4: INDOBERT EMBEDDING EXTRACTION
# ==========================================
print("⏳ Mengunduh dan menyiapkan Model IndoBERT...")
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
bert_model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1").to(device)

def get_bert_embeddings(texts, batch_size=16):
    embeddings = []
    bert_model.eval() # Set mode evaluasi
    
    with torch.no_grad(): # Matikan gradient agar hemat VRAM GPU
        for i in tqdm(range(0, len(texts), batch_size), desc="Ekstraksi Embedding"):
            batch_texts = texts[i:i+batch_size].tolist()
            
            # Tokenisasi
            inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
            
            # Pindahkan ke GPU
            inputs = {key: val.to(device) for key, val in inputs.items()}
            
            # Eksekusi Model
            outputs = bert_model(**inputs)
            
            # Ambil token [CLS] (Token pertama yang merepresentasikan seluruh kalimat)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
            
    return np.vstack(embeddings)

print("⏳ Mengekstrak makna teks menjadi vektor (GPU Bekerja)...")
X_embeddings = get_bert_embeddings(X_final)

print(f"✅ Ekstraksi Selesai! Bentuk matrix X: {X_embeddings.shape} (Baris x 768 Dimensi Makna)")

⏳ Mengunduh dan menyiapkan Model IndoBERT...
⏳ Mengekstrak makna teks menjadi vektor (GPU Bekerja)...


Ekstraksi Embedding: 100%|██████████| 52/52 [00:02<00:00, 18.52it/s]

✅ Ekstraksi Selesai! Bentuk matrix X: (830, 768) (Baris x 768 Dimensi Makna)


# MODELING & TUNING

In [5]:
# ==========================================
# CELL 5: TRAINING & MLFLOW TRACKING
# ==========================================
# 1. Split Data
X_train, X_test, y_train, y_test = train_test_split(X_embeddings, Y_final, test_size=0.2, random_state=42)

# 2. Inisialisasi DagsHub
dagshub.init(repo_owner='NazeeraAlthea', repo_name='exigen-smart-maintenance', mlflow=True)

# 3. Mulai Tracking Eksperimen
with mlflow.start_run(run_name="Eksperimen_2_IndoBERT_Augmented"):
    
    # 4. Inisialisasi Model Klasifikasi (Hybrid)
    clf = MultiOutputClassifier(RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42))
    
    # 5. Training
    print("🤖 Mulai melatih Random Forest dengan otak IndoBERT...")
    clf.fit(X_train, y_train)
    
    # 6. Prediksi & Evaluasi
    y_pred = clf.predict(X_test)
    
    exact_match = np.all(y_pred == y_test.values, axis=1).mean()
    print(f"\n🎯 Exact Match Ratio: {exact_match:.2%}")
    mlflow.log_metric("exact_match_ratio", exact_match)
    
    print("-" * 30)
    for i, col in enumerate(kolom_target):
        acc = accuracy_score(y_test.iloc[:, i], y_pred[:, i])
        print(f"🔸 Akurasi {col.upper()}: {acc:.2%}")
        mlflow.log_metric(f"accuracy_{col}", acc)
        
    # 7. Catat Parameter & Simpan
    mlflow.log_param("Feature_Extractor", "IndoBERT (indobenchmark/indobert-base-p1)")
    mlflow.log_param("Augmentation", "Contextual Word Embedding (nlpaug)")
    mlflow.log_param("Model", "Random Forest")
    mlflow.sklearn.log_model(clf, "model_indobert_rf")
    
    print("\n✅ Cell 5 Selesai: Model Hybrid sangat tangguh berhasil disimpan ke DagsHub MLflow!")

Accessing as NazeeraAlthea

Initialized MLflow to track repo "NazeeraAlthea/exigen-smart-maintenance"

Repository NazeeraAlthea/exigen-smart-maintenance initialized!

🤖 Mulai melatih Random Forest dengan otak IndoBERT...

🎯 Exact Match Ratio: 14.46%
------------------------------
🔸 Akurasi KATEGORI_ASET: 57.23%
🔸 Akurasi SEVERITY: 53.01%
🔸 Akurasi ROOT_CAUSE: 50.60%
🔸 Akurasi JENIS_KERUSAKAN: 60.24%


2026/05/15 17:08:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 17:08:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Cell 5 Selesai: Model Hybrid sangat tangguh berhasil disimpan ke DagsHub MLflow!
🏃 View run Eksperimen_2_IndoBERT_Augmented at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/0/runs/7c2aa751e65a4fe8bdea13de8d211e9e
🧪 View experiment at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/0
